<a href="https://colab.research.google.com/github/LuizOgata/analisador-lexico/blob/main/AnalisadorLexicoIngressos_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# TOKENS
# ============================================================

TOKENS = [
    # Palavras reservadas
    ("INGRESSO",      r"\bingresso\b", re.IGNORECASE),
    ("SETOR",         r"\bsetor\b", re.IGNORECASE),
    ("LOTE",          r"\blote\b", re.IGNORECASE),
    ("MEIA",          r"\bmeia\b", re.IGNORECASE),
    ("DATA",          r"\bdata\b", re.IGNORECASE),

    # Literais específicos
    ("VALOR",         r"R\$\s*\d+(?:,\d{2})", 0),
    ("DATA_LITERAL",  r"\d{2}/\d{2}/\d{4}", 0),
    ("CODIGO_EVENTO", r"EVT-\d{4,6}", re.IGNORECASE),

    # Outros tokens
    ("MULTIPLICADOR", r"x\b", re.IGNORECASE),
    ("STRING",        r'"[^"\n]*"', 0),
    ("IDENTIFICADOR", r"[A-Za-zÀ-ÿ][A-Za-zÀ-ÿ0-9_]*", 0),
    ("NUMERO",        r"\d+", 0),
    ("SIMBOLO",       r"[.,]", 0),
]


# ============================================================
# CONFLITO DE PRIORIDADE
# ============================================================
#
# CONFLITO:
# A entrada "20/07/2026" poderia ser inicialmente interpretada
# como vários números separados por "/".
#
# SOLUÇÃO:
# DATA_LITERAL aparece antes de NUMERO na lista de tokens.
# Assim, o analisador tenta reconhecer primeiro a data completa.
#
# Outro conflito:
# "R$ 180,00" poderia ser dividido em identificadores, números
# e símbolos. VALOR possui prioridade maior e captura a expressão
# monetária inteira.
#
# Portanto, regras específicas aparecem antes das genéricas.
# ============================================================


class ErroLexico:
    def __init__(self, mensagem, linha, coluna, dica):
        self.mensagem = mensagem
        self.linha = linha
        self.coluna = coluna
        self.dica = dica

    def __str__(self):
        return (
            f"Linha {self.linha}, coluna {self.coluna}: "
            f"{self.mensagem}\n"
            f"💡 Dica: {self.dica}"
        )


class AnalisadorLexico:

    def __init__(self):
        self.erros = []
        self.tokens = []

    def descobrir_linha_coluna(self, texto, posicao):
        linha = texto.count("\n", 0, posicao) + 1

        ultima_quebra = texto.rfind("\n", 0, posicao)

        if ultima_quebra == -1:
            coluna = posicao + 1
        else:
            coluna = posicao - ultima_quebra

        return linha, coluna

    def analisar(self, texto):

        self.erros = []
        self.tokens = []

        pos = 0

        while pos < len(texto):

            # ------------------------------------------------
            # Espaços em branco
            # ------------------------------------------------
            if texto[pos].isspace():
                pos += 1
                continue

            # ------------------------------------------------
            # Comentários
            # ------------------------------------------------
            if texto[pos] == "#":
                while pos < len(texto) and texto[pos] != "\n":
                    pos += 1
                continue

            encontrou = False

            # ------------------------------------------------
            # Tenta cada token de acordo com a prioridade
            # ------------------------------------------------
            for nome, regex, flags in TOKENS:

                padrao = re.compile(regex, flags)
                match = padrao.match(texto, pos)

                if match:

                    lexema = match.group()

                    linha, coluna = self.descobrir_linha_coluna(
                        texto, pos
                    )

                    self.tokens.append({
                        "Token": nome,
                        "Lexema": lexema,
                        "Linha": linha,
                        "Coluna": coluna
                    })

                    pos = match.end()
                    encontrou = True
                    break

            # ------------------------------------------------
            # Erro léxico
            # ------------------------------------------------
            if not encontrou:

                caractere = texto[pos]

                linha, coluna = self.descobrir_linha_coluna(
                    texto, pos
                )

                # Dicas específicas do domínio
                if caractere == "R":
                    dica = (
                        "Para valores de ingresso, use o formato "
                        "R$ 180,00."
                    )

                elif caractere == "/":
                    dica = (
                        "Para datas de eventos, use o formato "
                        "DD/MM/AAAA, por exemplo 20/07/2026."
                    )

                elif caractere == "@":
                    dica = (
                        "Verifique o nome do evento, setor ou código "
                        "do ingresso. Caracteres '@' não são aceitos."
                    )

                else:
                    dica = (
                        "Verifique o nome do evento, setor ou código "
                        "especial do ingresso."
                    )

                self.erros.append(
                    ErroLexico(
                        f"Caractere inesperado '{caractere}'",
                        linha,
                        coluna,
                        dica
                    )
                )

                pos += 1

        return self.tokens, self.erros


# ============================================================
# INTERFACE
# ============================================================

entrada = widgets.Textarea(
    value=(
        'INGRESSO 2x "Festival de Inverno" '
        'SETOR pista LOTE 2 MEIA R$ 180,00 '
        'DATA 20/07/2026'
    ),
    placeholder="Digite a entrada da mini-linguagem...",
    description="Entrada:",
    layout=widgets.Layout(
        width="100%",
        height="180px"
    )
)

botao = widgets.Button(
    description="Analisar entrada",
    button_style="primary",
    icon="search"
)

saida = widgets.HTML()

tabela_saida = widgets.Output()


def executar_analise(_):

    analisador = AnalisadorLexico()

    tokens, erros = analisador.analisar(
        entrada.value
    )

    # --------------------------------------------------------
    # Texto colorido
    # --------------------------------------------------------

    texto_original = html.escape(entrada.value)

    if erros:
        status = (
            '<div style="background:#ffe5e5; padding:12px; '
            'border-radius:8px; color:#a00000;">'
            '<b>❌ Entrada contém erros léxicos.</b>'
            '</div>'
        )

        erros_html = "<br>".join(
            [
                f"⚠️ Linha {e.linha}, coluna {e.coluna}: "
                f"{html.escape(e.mensagem)}"
                f"<br>&nbsp;&nbsp;&nbsp;💡 "
                f"{html.escape(e.dica)}"
                for e in erros
            ]
        )

    else:

        status = (
            '<div style="background:#e5ffe5; padding:12px; '
            'border-radius:8px; color:#006400;">'
            '<b>✅ Entrada válida! Nenhum erro léxico encontrado.</b>'
            '</div>'
        )

        erros_html = ""

    saida.value = f"""
    <div style="
        font-family:monospace;
        background:#202124;
        color:#ffffff;
        padding:15px;
        border-radius:8px;
        margin-top:10px;
    ">
        <b>Entrada analisada:</b><br><br>
        {texto_original}
    </div>

    <br>

    {status}

    <br>

    {erros_html}
    """

    # --------------------------------------------------------
    # Tabela
    # --------------------------------------------------------

    with tabela_saida:

        tabela_saida.clear_output()

        if tokens:

            df = pd.DataFrame(tokens)

            display(
                df.style.set_properties(
                    **{
                        "text-align": "left",
                        "font-size": "12pt"
                    }
                )
            )

        else:

            print("Nenhum token reconhecido.")


botao.on_click(executar_analise)


display(
    widgets.VBox([
        widgets.HTML(
            "<h2>🎟️ Analisador Léxico — Ingressos de Eventos</h2>"
        ),

        widgets.HTML(
            "<p>Mini-linguagem inspirada em sistemas de "
            "venda de ingressos.</p>"
        ),

        entrada,

        botao,

        saida,

        widgets.HTML("<h3>📋 Tabela de Tokens</h3>"),

        tabela_saida
    ])
)

INGRESSO 2x "Festival de Inverno" SETOR pista LOTE 2 MEIA R$ 180,00 DATA 20/07/2026

INGRESSO 1x "Show de Rock" SETOR premium LOTE 1 R$ 350,00 DATA 15/08/2026


INGRESSO 3x "Festival de Verão" SETOR arquibancada LOTE 3 MEIA R$ 95,50 DATA 10/01/2027

Casos inválidos
Inválido 1 — caractere não permitido
INGRESSO 2x "Festival @ Inverno" SETOR pista R$ 180,00 DATA 20/07/2026

Inválido 2 — data incorreta
INGRESSO 2x "Festival de Inverno" SETOR pista DATA 20/07/26